# LIORA — Analyse de sentiment sur avis Trustpilot

**Objectif** : classifier les verbatim clients (positif / neutre / négatif) à partir des avis scrapés.

**Label proxy** : dérivé de la note 1–5 (weak supervision pédagogique).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from liora.ml.sentiment import build_training_frame, train_sentiment_model, predict_sentiment
from liora.config import RAW_DIR

sns.set_theme(style='whitegrid')

In [ ]:
df = pd.read_csv(RAW_DIR / 'avis_trustpilot.csv')
print(f"{len(df)} avis — {df['company_name'].nunique()} entreprises")
df.head()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df.groupby('company_name')['rating'].mean().sort_values().plot(kind='barh', ax=ax[0], color='steelblue')
ax[0].set_title('Note moyenne par entreprise')
df['rating'].value_counts().sort_index().plot(kind='bar', ax=ax[1], color='coral')
ax[1].set_title('Distribution des notes')
plt.tight_layout()
plt.show()

In [ ]:
frame = build_training_frame(df)
frame['sentiment'].value_counts().plot(kind='pie', autopct='%1.1f%%', title='Répartition sentiments (proxy)')
plt.ylabel('')
plt.show()

In [ ]:
model, metrics, _ = train_sentiment_model(df)
print('F1 macro:', round(metrics['f1_macro'], 3))
print(metrics['report'])

In [ ]:
# Exemples de prédictions sur avis récents
samples = df.groupby('company_name').head(2)['text'].tolist()
for text, pred in zip(samples, predict_sentiment(model, samples)):
    print(f"[{pred['sentiment']}] {text[:80]}...")

## MLflow

Le versioning est géré par `python -m liora.ml.train` (URI : http://localhost:5000).